# Этап 1. EDA — часть 2: суммы, время/velocity, коридоры, счета

**Статус:** первая часть (загрузка, `df.info()`, пропуски, дисбаланс, типологии) уже сделана.
Эта тетрадка — продолжение. Она самодостаточна: если запустить с самого начала,
она пересчитает и то, что уже было, но быстро (данные берутся из кэша).

## Что делаем дальше и зачем (логика AML-расследования)

Отмывание денег — это **не одна транзакция**, это **сценарий (typology)** из нескольких
шагов. Классическая схема из трёх стадий:

| Стадия | Что происходит | Какие типологии SAML-D |
|---|---|---|
| **Placement** (размещение) | грязные деньги/наличные вводятся в финансовую систему | `Cash_Withdrawal`, `Deposit-Send`, `Structuring`, `Smurfing` |
| **Layering** (наслоение) | деньги запутываются: много переводов, цепочки, круговые схемы | `Fan_In`, `Fan_Out`, `Layered_Fan_In/Out`, `Cycle`, `Bipartite`, `Gather-Scatter`, `Scatter-Gather`, `Stacked Bipartite` |
| **Integration** (интеграция) | деньги возвращаются владельцу уже «чистыми» | `Single_large`, `Over-Invoicing`, `Behavioural_Change_1/2` |

Задача этого EDA — **увидеть в данных следы каждого из этих шагов** и понять,
какие признаки потом строить (Этап 2) и какие правила писать (Этап 3).

План:
1. Напоминание: дисбаланс и типологии *(уже делали — сжато)*
2. **Суммы:** structuring-тест у порога $10 000, «круглые» суммы, сумма относительно обычного поведения счёта
3. **Время и velocity:** часы, выходные, серии транзакций, интервалы между платежами
4. **География/коридоры:** страны, кросс-бордер, несовпадение валют, типы платежей
5. **Счета и граф (лёгкая версия):** fan-in/fan-out, транзитные счета, mutual-пары (циклы)
6. **Качество данных** и сохранение итогов

> Каждый блок заканчивается вопросом **«что это значит для бизнеса»** — именно эти
> выводы потом перекочуют в README и в рассказ на собеседовании.

In [ ]:
# ============================================================
# 0. НАСТРОЙКА
# ============================================================
import sys, warnings, gc
from pathlib import Path

# Находим корень проекта (папку, в которой лежит src/), чтобы импорт src.* работал
# независимо от того, откуда запущен ноутбук (из notebooks/ или из корня).
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "data_loader.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import (
    load_dataset, memory_mb, class_balance, missing_report, save_table,
    COL_TIME, COL_DATE, COL_SENDER, COL_RECEIVER, COL_AMOUNT, COL_PAY_CUR,
    COL_REC_CUR, COL_SENDER_LOC, COL_RECEIVER_LOC, COL_PAY_TYPE,
    COL_TARGET, COL_LAUND_TYPE, STRUCTURING_THRESHOLD, FIGURES_DIR, PROCESSED_DIR,
)
from src.eda_utils import (
    rate_by_group, rate_by_bucket, compare_band, two_proportion_ztest,
    save_fig, bucketize, categories_equal,
)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
SAMPLE_PLOT = 200_000   # сколько точек брать на плотные графики (kde/scatter)
COLOR_OK, COLOR_BAD = "#4C72B0", "#C44E52"

print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
# ============================================================
# 1. ЗАГРУЗКА (с кэшем!)
# ============================================================
# ПЕРВЫЙ запуск: читает CSV (~1-3 мин на 9.5 млн строк) и сохраняет кэш
#                в data/processed/saml_d_prepared.parquet
# ВТОРОЙ и далее: читает кэш за секунды.
#
# Если менял логику подготовки в src/data_loader.py — запусти с force_rebuild=True.
df = load_dataset(use_cache=True, force_rebuild=False, verbose=True)

BASE_RATE = float(df[COL_TARGET].mean())   # доля отмывания = наш «уровень шума»
print(f"\nСтрок: {len(df):,} | Память: {memory_mb(df):,.0f} MB")
print(f"Base rate (доля Is_laundering=1): {BASE_RATE * 100:.4f}%")
print(f"Дисбаланс: 1 отмывание на {int(1 / BASE_RATE):,} транзакций")
if "date" in df.columns:
    print(f"Период: {df['date'].min().date()} .. {df['date'].max().date()} "
          f"({df['date'].nunique()} дней)")
display(df.head())
print("\nПропуски по колонкам:")
display(missing_report(df))

In [ ]:
# ============================================================
# 1.1 Производные колонки, нужные и в EDA, и дальше в признаках
# ============================================================
# Почему считаем здесь: эти флаги понадобятся в каждом втором графике.
# (На Этапе 2 мы соберём все признаки системно и аккуратно, здесь — «черновик для глаз».)

# Кросс-бордер: деньги уходят в другую юрисдикцию. Классический фактор риска:
# разные страны = разный уровень AML-контроля и разные пороги отчётности.
# categories_equal сравнивает числовые коды категорий — без выделения памяти
# под миллионы строковых значений (на 9.5 млн строк это разница в минуты и ГБ).
df["is_cross_border"] = (~categories_equal(df[COL_SENDER_LOC], df[COL_RECEIVER_LOC])).astype("int8")

# Несовпадение валют: платим в одной, получаем в другой. Легально (конвертация),
# но это ещё и самый простой способ скрыть след и «сбить» сумму (layering).
df["is_currency_mismatch"] = (~categories_equal(df[COL_PAY_CUR], df[COL_REC_CUR])).astype("int8")

# Процентиль суммы ВНУТРИ своей валюты.
# Зачем: Amount в датасете в разных валютах (USD, EUR, Yuan, Rupee...),
# поэтому сравнивать «9 000 долларов» и «9 000 рупий» нельзя. Процентиль приводит
# всё к одной шкале: «насколько крупная эта транзакция для своей валюты».
df["amount_pct_in_currency"] = (
    df.groupby(COL_PAY_CUR, observed=True)[COL_AMOUNT].rank(pct=True) * 100
).astype("float32")

print("Добавлены: is_cross_border, is_currency_mismatch, amount_pct_in_currency")
print(f"Память теперь: {memory_mb(df):,.0f} MB")

---
# 2. Напоминание: дисбаланс классов и типологии

*(этот блок уже делался — оставил сжато, чтобы тетрадка была цельной)*

**Почему это ключевая особенность AML, а не «просто несбалансированные данные»:**

* Из 10 000 транзакций только ~10 — отмывание. Значит accuracy 99.9% = модель,
  которая предсказывает «всё чисто» и **не имеет никакой ценности**.
* Мало positives → нельзя просто «сэмплировать»: теряются редкие типологии
  (`Over-Invoicing` — всего 54 случая на 9.5 млн).
* Отсюда все решения дальше: метрики **PR-AUC / recall@K**, а не accuracy;
  разбиение на train/test **по времени**; порог модели выбираем по бизнес-бюджету
  на проверку алертов (у аналитика конечное число рук).

In [ ]:
# ---- Дисбаланс -------------------------------------------------------
bal = class_balance(df[COL_TARGET])
print(f"Всего транзакций:   {bal['n']:,}")
print(f"Отмывание (=1):     {bal['positives']:,}  ({bal['base_rate'] * 100:.4f}%)")
print(f"Легальные (=0):     {bal['negatives']:,}")
print(f"Дисбаланс:          1 : {bal['imbalance_ratio']:,.0f}")

# ---- Типологии -------------------------------------------------------
laund = df[df[COL_TARGET] == 1]        # только «грязные» транзакции
typ = (
    laund[COL_LAUND_TYPE]
    .value_counts(dropna=False)
    .rename("n_transactions")
    .to_frame()
)
typ["pct_of_laundering"] = (typ["n_transactions"] / typ["n_transactions"].sum() * 100).round(2)

# Сколько уникальных счетов задействовано в каждой типологии — пригодится дальше
typ["n_senders"] = (
    laund.groupby(COL_LAUND_TYPE, observed=True)[COL_SENDER].nunique().reindex(typ.index)
)
display(typ)

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

# Левый график: баланс классов (лог. шкала, иначе столбец отмывания не виден)
axes[0].bar(["Легальные (0)", "Отмывание (1)"],
            [bal["negatives"], bal["positives"]], color=[COLOR_OK, COLOR_BAD])
axes[0].set_yscale("log")
axes[0].set_title("Баланс классов (лог. шкала)")
axes[0].set_ylabel("Число транзакций")
for i, v in enumerate([bal["negatives"], bal["positives"]]):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontweight="bold")

# Правый график: раскладка по типологиям
axes[1].barh(typ.index[::-1], typ["n_transactions"].values[::-1], color=COLOR_BAD, alpha=.85)
axes[1].set_title("Типологии отмывания (число транзакций)")
axes[1].set_xlabel("n транзакций")
save_fig("02_class_balance_and_typologies", fig)
plt.show()

---
# 3. Анализ сумм (Amount)

## Бизнес-контекст: почему сумма — признак №1

Почти все пороги в AML привязаны к сумме:

* **CTR (Currency Transaction Report)** — в США банк обязан сообщить о любой
  операции наличными свыше **$10 000**. В ЕС порог схожий (€10 000–15 000),
  в Грузии/Турции — свои значения.
* Преступники это знают → **структурирование (structuring / smurfing)**: дробят
  $100 000 на 12 переводов по $9 500, чтобы ни один не дотянул до порога.
* Отсюда гипотеза, которую сейчас проверим:
  **доля отмывания в «полосе» чуть ниже $10 000 аномально выше, чем в среднем.**

В SAML-D это одна из самых массовых типологий (`Structuring` — 1870 транзакций,
`Smurfing` — 932), поэтому сигнал должен быть виден.

In [ ]:
# ---- 3.1 Базовая статистика суммы по классам -------------------------
desc = df.groupby(COL_TARGET, observed=True)[COL_AMOUNT].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
)
desc.index = ["Легальные (0)", "Отмывание (1)"]
display(desc.T)

print("Как читать: если медиана/квантили у отмывания заметно отличаются — значит")
print("«подозрительные операции другого размера, чем обычные платежи».")

In [ ]:
# ---- 3.2 Распределение сумм: гистограммы на лог-шкале ----------------
# Почему log10: суммы распределены логнормально (много мелких, редкие огромные).
# На обычной шкале график превращается в один столбик у нуля.

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

samples = {}
for cls, name, color in [(0, "Легальные", COLOR_OK), (1, "Отмывание", COLOR_BAD)]:
    s = df.loc[df[COL_TARGET] == cls, COL_AMOUNT]
    s = s.sample(min(len(s), SAMPLE_PLOT), random_state=RANDOM_STATE)
    samples[cls] = np.log10(s.clip(lower=0.01))
    axes[cls].hist(samples[cls], bins=70, color=color, alpha=.85)
    axes[cls].set_title(f"{name}: распределение log10(Amount)")
    axes[cls].set_xlabel("log10(Amount)")
    axes[cls].set_ylabel("число транзакций")

# Третий график: два распределения на одной оси (нормированные на долю) —
# так видно СДВИГ: где именно отмывание «гуще» обычных платежей.
axes[2].hist(samples[0], bins=70, density=True, alpha=.55, label="Легальные", color=COLOR_OK)
axes[2].hist(samples[1], bins=70, density=True, alpha=.55, label="Отмывание", color=COLOR_BAD)
axes[2].set_title("Сравнение форм распределения")
axes[2].set_xlabel("log10(Amount)")
axes[2].legend()
save_fig("03_amount_distributions", fig)
plt.show()

In [ ]:
# ---- 3.3 Boxplot по классам (лог. шкала по Y) ------------------------
# Boxplot на 9.5 млн точек рисуется долго -> берём выборку.
plot_df = pd.DataFrame({
    "class": df[COL_TARGET].map({0: "Легальные", 1: "Отмывание"}).astype(str),
    "Amount": df[COL_AMOUNT],
}).sample(min(len(df), 400_000), random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=plot_df, x="class", y="Amount", hue="class", legend=False,
            showfliers=False,
            palette={"Легальные": COLOR_OK, "Отмывание": COLOR_BAD}, ax=ax)
ax.set_yscale("log")
ax.set_title("Amount по классам (медиана и квартили, лог-шкала)")
ax.set_xlabel("")
save_fig("04_amount_boxplot_by_class", fig)
plt.show()

In [ ]:
# ---- 3.4 Процентиль суммы внутри валюты ------------------------------
# Ещё раз зачем: Amount в долларах и Amount в рупиях несравнимы напрямую.
# Процентиль внутри валюты = «насколько это крупный платёж для своей валюты».

pct_desc = df.groupby(COL_TARGET, observed=True)["amount_pct_in_currency"].describe(
    percentiles=[.05, .25, .5, .75, .95]
)
pct_desc.index = ["Легальные (0)", "Отмывание (1)"]
display(pct_desc)

# Средний процентиль по каждой типологии — сразу видно «крупные» и «мелкие» схемы
typ_amount = (
    laund.groupby(COL_LAUND_TYPE, observed=True)
    .agg(n=("amount_pct_in_currency", "size"),
         median_pct=("amount_pct_in_currency", "median"),
         median_amount=(COL_AMOUNT, "median"),
         max_amount=(COL_AMOUNT, "max"))
    .sort_values("median_pct", ascending=False)
)
display(typ_amount)

## 3.5 Главный тест: «прижимание к порогу $10 000» (structuring)

**Как проверяем (это важно уметь объяснить на собеседовании):**

1. Режем диапазон $0–20 000 на корзины по $1 000.
2. В каждой корзине считаем **долю отмывания** и **lift** = доля / base rate.
3. Отдельно сравниваем полосу **$9 000–10 000** со всеми остальными транзакциями
   через **z-тест двух долей** — чтобы доказать, что всплеск не случаен.
4. Сравниваем полосу **ниже** порога с полосой **сразу выше** ($10 000–11 000):
   если structuring есть, справа от порога должен быть резкий обвал («обрыв»).

**Ожидание:** график lift растёт к $10 000 слева и падает сразу после него.

In [ ]:
# ---- 3.5.1 Rate и lift по корзинам сумм ------------------------------
bins = np.arange(0, 20_001, 1_000)
tmp = df[[COL_AMOUNT, COL_TARGET]].copy()
tmp["bin"] = pd.cut(tmp[COL_AMOUNT], bins=bins, right=False)

amt_tbl = tmp.groupby("bin", observed=True)[COL_TARGET].agg(n="size", n_laundering="sum")
amt_tbl["rate_pct"] = amt_tbl["n_laundering"] / amt_tbl["n"] * 100
amt_tbl["lift"] = (amt_tbl["n_laundering"] / amt_tbl["n"]) / BASE_RATE
amt_tbl["bin_mid"] = [iv.left + 500 for iv in amt_tbl.index]
display(amt_tbl[["n", "n_laundering", "rate_pct", "lift", "bin_mid"]].round(3))

In [ ]:
# ---- 3.5.2 График: «ступенька» у порога ------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))

# Слева: объём транзакций по корзинам (сколько платежей в каждой полосе)
axes[0].bar(amt_tbl["bin_mid"], amt_tbl["n"], width=800, color=COLOR_OK, alpha=.85)
axes[0].axvline(STRUCTURING_THRESHOLD, color=COLOR_BAD, ls="--", lw=2,
                label=f"порог ${STRUCTURING_THRESHOLD:,.0f}")
axes[0].set_title("Сколько транзакций в каждой полосе сумм")
axes[0].set_xlabel("Amount"); axes[0].set_ylabel("n транзакций"); axes[0].legend()

# Справа: lift — во сколько раз полоса «грязнее» среднего
axes[1].bar(amt_tbl["bin_mid"], amt_tbl["lift"], width=800, color=COLOR_BAD, alpha=.85)
axes[1].axvline(STRUCTURING_THRESHOLD, color="black", ls="--", lw=2)
axes[1].axhline(1.0, color="grey", lw=1, label="lift = 1 (средний уровень)")
axes[1].set_title("Lift отмывания по полосам сумм\n(столбец прямо перед порогом = structuring)")
axes[1].set_xlabel("Amount"); axes[1].set_ylabel("lift к base rate"); axes[1].legend()
save_fig("05_structuring_threshold_lift", fig)
plt.show()

In [ ]:
# ---- 3.5.3 Статистический тест: полоса $9 000–10 000 против остальных -
band_lo = STRUCTURING_THRESHOLD * 0.90     # 9 000
band_hi = STRUCTURING_THRESHOLD            # 10 000

near_mask = df[COL_AMOUNT].between(band_lo, band_hi, inclusive="left")   # [9000, 10000)
above_mask = df[COL_AMOUNT].between(band_hi, band_hi * 1.10, inclusive="left")  # [10000, 11000)

print(f"Полоса НИЖЕ порога  [{band_lo:,.0f} .. {band_hi:,.0f})")
_ = compare_band(df, near_mask,
                 label_in=f"сумма {band_lo:,.0f}-{band_hi:,.0f}",
                 label_out="все остальные суммы")

print("\n" + "=" * 70)
print("Сравнение полосы НИЖЕ порога и полосы СРАЗУ ВЫШЕ порога:")
x_near, n_near = int(df.loc[near_mask, COL_TARGET].sum()), int(near_mask.sum())
x_above, n_above = int(df.loc[above_mask, COL_TARGET].sum()), int(above_mask.sum())
z, p = two_proportion_ztest(x_near, n_near, x_above, n_above)
print(f"  ниже порога: {x_near:,} / {n_near:,} = {x_near / n_near * 100:.4f}%")
print(f"  выше порога: {x_above:,} / {n_above:,} = {x_above / n_above * 100:.4f}%")
print(f"  z = {z:,.2f}, p-value = {p:.3e}")

> **Как интерпретировать результат.**
> Если lift в полосе $9 000–10 000 > 1.5–2 и p-value < 0.05 — гипотеза подтверждена:
> преступники в данных реально «прижимаются» к порогу. Это готовый аргумент
> для Этапа 3: правило «сумма в 90–100% от порога» — не выдумка, а подтверждённый
> статистикой паттерн.
>
> Если lift ~ 1 — тоже нормально: значит structuring в этой версии датасета
> смоделирован через **количество** платежей, а не через сумму, и надо проверять
> гипотезу «много платежей 9–10k за день» (блок 4.6).

## 3.6 «Круглые» суммы

**Бизнес-логика:** легальные платежи обычно «рваные» ($1 234.56 — чек, покупка).
А переводы между своими счетами при layering часто делают круглыми
($50 000, $100 000) — так проще делить и контролировать поток.
Проверим, правда ли это на наших данных.

In [ ]:
# ---- 3.6 Круглые суммы ----------------------------------------------
amt = df[COL_AMOUNT].round(2)   # округление до центов: снимаем артефакты float32

roundness = {
    "кратно 10 000": (amt % 10_000 == 0),
    "кратно 1 000":  (amt % 1_000 == 0),
    "кратно 100":    (amt % 100 == 0),
    "кратно 10":     (amt % 10 == 0),
}

rows = []
for name, m in roundness.items():
    n_in, x_in = int(m.sum()), int(df.loc[m, COL_TARGET].sum())
    n_out, x_out = int((~m).sum()), int(df.loc[~m, COL_TARGET].sum())
    z, p = two_proportion_ztest(x_in, n_in, x_out, n_out)
    rate_in = x_in / n_in * 100 if n_in else np.nan
    rate_out = x_out / n_out * 100 if n_out else np.nan
    rows.append({
        "проверка": name,
        "n_transactions": n_in,
        "доля_таких_%": n_in / len(df) * 100,
        "rate_внутри_%": rate_in,
        "rate_вне_%": rate_out,
        "lift": rate_in / rate_out if rate_out else np.nan,
        "p_value": p,
    })

round_tbl = pd.DataFrame(rows).set_index("проверка")
display(round_tbl)

In [ ]:
# ---- 3.7 Суммы по типологиям: какая схема какого размера -------------
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

order = typ.index.tolist()   # порядок от самой частой типологии к редкой

sns.boxplot(data=laund[[COL_LAUND_TYPE, COL_AMOUNT]], x=COL_LAUND_TYPE, y=COL_AMOUNT,
            order=order, showfliers=False, color=COLOR_BAD, ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Amount по типологиям (лог-шкала)")
axes[0].set_xlabel(""); axes[0].set_ylabel("Amount")
axes[0].tick_params(axis="x", rotation=75)

sns.boxplot(data=laund[[COL_LAUND_TYPE, "amount_pct_in_currency"]],
            x=COL_LAUND_TYPE, y="amount_pct_in_currency",
            order=order, showfliers=False, color="#55A868", ax=axes[1])
axes[1].set_title("Процентиль суммы внутри валюты по типологиям")
axes[1].set_xlabel(""); axes[1].set_ylabel("процентиль, %")
axes[1].tick_params(axis="x", rotation=75)
save_fig("06_amount_by_typology", fig)
plt.show()

## 3.8 Поведенческий сдвиг: сумма относительно обычного поведения счёта

**Бизнес-логика (типологии `Behavioural_Change_1/2`):**
преступник сначала «прогревает» счёт — ведёт себя нормально несколько месяцев,
а потом резко меняет паттерн: суммы вырастают в разы.
Банки ловят это правилом «транзакция сильно отклоняется от исторического
профиля клиента» — это и есть **behavioural profiling**, основа современного мониторинга.

Метрика: `Amount / медианная сумма этого отправителя`.

In [ ]:
# ---- 3.8 Отклонение от собственной медианы ---------------------------
# transform("median") — посчитать медиану по каждому счёту и «развернуть» её
# обратно на каждую строку. Без циклов; на 9.5 млн строк — секунды/десятки секунд.
df["sender_median_amount"] = df.groupby(COL_SENDER, observed=True)[COL_AMOUNT].transform("median")
df["amount_vs_own_median"] = df[COL_AMOUNT] / df["sender_median_amount"].replace(0, np.nan)

df["_ratio_bin"] = bucketize(
    df["amount_vs_own_median"],
    bins=[0, 0.5, 0.9, 1.1, 2, 5, 10, 50, np.inf],
    labels=["<0.5x", "0.5-0.9x", "0.9-1.1x (обычно)", "1.1-2x", "2-5x", "5-10x", "10-50x", ">50x"],
)

beh = rate_by_bucket(df, "_ratio_bin")
display(beh.round(3))

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(beh.index.astype(str), beh["lift"], color="#8172B2")
ax.axhline(1.0, color="grey", lw=1)
ax.set_title("Lift отмывания по отношению суммы к обычной сумме счёта")
ax.set_ylabel("lift к base rate")
ax.tick_params(axis="x", rotation=30)
save_fig("07_behavioural_deviation_lift", fig)
plt.show()

---
# 4. Время и velocity (скорость операций)

## Бизнес-контекст

**Velocity** в AML — это «сколько операций прошло через счёт за период»
(час / день / неделя). Это второй по силе признак после суммы, потому что
типология `Smurfing` — это дословно «серия однотипных мелких платежей за короткое время».

Что именно ищем:

| Метрика | Что ловит |
|---|---|
| Час суток | отмывание часто идёт ночью и «под закрытие» дня |
| День недели | всплеск в выходные, когда живой контроль слабее |
| Число транзакций счёта за день | fan-out / smurfing |
| Интервал между транзакциями (inter-arrival) | серии-«очереди» платежей |
| Число транзакций счёта за час | «burst» — резкий залп |

> **Важный вывод для Этапа 4 (ML):** у данных есть временная структура.
> Значит train/test **нельзя** перемешивать случайно — только **разбиение по времени**
> (учимся на прошлом, проверяем на будущем). Иначе модель «увидит будущее» и
> покажет нереалистично красивый результат. Это частая ошибка джунов.

In [ ]:
# ---- 4.1 Час суток ---------------------------------------------------
hour_tbl = rate_by_group(df, "hour", min_support=1_000, sort_by="hour", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))

axes[0].bar(hour_tbl.index, hour_tbl["n"], color=COLOR_OK, alpha=.85)
axes[0].set_title("Объём транзакций по часам суток")
axes[0].set_xlabel("час"); axes[0].set_ylabel("n транзакций")

axes[1].plot(hour_tbl.index, hour_tbl["rate_pct"], marker="o", color=COLOR_BAD, lw=2,
             label="доля отмывания, %")
axes[1].axhline(BASE_RATE * 100, color="grey", ls="--",
                label=f"base rate = {BASE_RATE * 100:.4f}%")
axes[1].set_title("Доля отмывания по часам суток")
axes[1].set_xlabel("час"); axes[1].set_ylabel("rate, %")
axes[1].legend()
save_fig("08_hour_of_day", fig)
plt.show()

display(hour_tbl.sort_values("lift", ascending=False).head(5).round(3))

In [ ]:
# ---- 4.2 День недели и календарный тренд -----------------------------
dow_names = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]
dow = rate_by_group(df, "day_of_week", sort_by="day_of_week", ascending=True)
dow.index = [dow_names[int(i)] for i in dow.index]
display(dow.round(3))

# Тренд по дням: нужен, чтобы понять, есть ли «сезонность» и как резать train/test
daily = df.groupby("date", observed=True)[COL_TARGET].agg(n="size", n_laundering="sum")
daily["rate_pct"] = daily["n_laundering"] / daily["n"] * 100

fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
axes[0].plot(daily.index, daily["n"], color=COLOR_OK, lw=1)
axes[0].set_title("Число транзакций в день")
axes[0].set_ylabel("n в день")
axes[1].plot(daily.index, daily["rate_pct"], color=COLOR_BAD, lw=1)
axes[1].axhline(BASE_RATE * 100, color="grey", ls="--")
axes[1].set_title("Доля отмывания в день (rate, %)")
axes[1].set_ylabel("rate, %")
save_fig("09_daily_trend", fig)
plt.show()

print(f"Дней в данных: {len(daily)}")
print("Вывод для разбиения: если rate меняется от месяца к месяцу -> только time-based split.")

In [ ]:
# ---- 4.3 Velocity: сколько транзакций делает счёт за день -------------
# Группируем по (счёт, дата): это и есть «дневная активность клиента».
sday = df.groupby([COL_SENDER, "date"], observed=True)[COL_TARGET].agg(n="size", n_laundering="sum")
sday.columns = ["n_txn", "n_laundering"]

sday["bucket"] = bucketize(
    sday["n_txn"],
    bins=[1, 2, 3, 4, 5, 10, 20, 50, np.inf],
    labels=["1", "2", "3", "4", "5-9", "10-19", "20-49", "50+"],
)

vel = sday.groupby("bucket", observed=True).agg(
    n_client_days=("n_txn", "size"),
    n_txn=("n_txn", "sum"),
    n_laundering=("n_laundering", "sum"),
)
vel["rate_pct"] = vel["n_laundering"] / vel["n_txn"] * 100
vel["lift"] = (vel["n_laundering"] / vel["n_txn"]) / BASE_RATE
display(vel.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].bar(vel.index.astype(str), vel["n_txn"], color=COLOR_OK)
axes[0].set_yscale("log")
axes[0].set_title("Сколько транзакций приходится на корзины дневной активности")
axes[0].set_xlabel("транзакций за день у счёта"); axes[0].set_ylabel("n транзакций (лог)")

axes[1].bar(vel.index.astype(str), vel["lift"], color=COLOR_BAD)
axes[1].axhline(1.0, color="grey", lw=1)
axes[1].set_title("Lift отмывания vs дневная активность счёта")
axes[1].set_xlabel("транзакций за день у счёта"); axes[1].set_ylabel("lift")
save_fig("10_velocity_per_day", fig)
plt.show()

del sday   # освобождаем память: 8 млн групповых строк больше не нужны
gc.collect()


In [ ]:
# ---- 4.4 Интервал между транзакциями одного счёта (inter-arrival) ----
# Сортируем операции каждого счёта по времени и берём разницу с предыдущей.
# Smurfing выглядит так: 20 платежей с интервалом 1-3 минуты.

gap = df[[COL_SENDER, "txn_ts", COL_TARGET]].copy()
gap = gap.sort_values([COL_SENDER, "txn_ts"], kind="mergesort")
gap["gap_min"] = gap.groupby(COL_SENDER, observed=True)["txn_ts"].diff().dt.total_seconds() / 60
gap = gap.dropna(subset=["gap_min"])
print(f"Пар «предыдущая -> текущая»: {len(gap):,}")

gap["bucket"] = bucketize(
    gap["gap_min"],
    bins=[0, 1, 5, 30, 120, 720, 1440, 10080, np.inf],
    labels=["<1 мин", "1-5 мин", "5-30 мин", "0.5-2 ч", "2-12 ч",
            "12-24 ч", "1-7 дней", ">7 дней"],
)
gap_tbl = rate_by_bucket(gap, "bucket")
display(gap_tbl.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].bar(gap_tbl.index.astype(str), gap_tbl["lift"], color="#CCB974")
axes[0].axhline(1.0, color="grey", lw=1)
axes[0].set_title("Lift отмывания vs интервал между транзакциями счёта")
axes[0].set_ylabel("lift"); axes[0].tick_params(axis="x", rotation=30)

for cls, name, color in [(0, "Легальные", COLOR_OK), (1, "Отмывание", COLOR_BAD)]:
    s = gap.loc[gap[COL_TARGET] == cls, "gap_min"]
    s = s.sample(min(len(s), SAMPLE_PLOT), random_state=RANDOM_STATE)
    axes[1].hist(np.log10(s.clip(lower=0.1)), bins=60, density=True,
                 alpha=.55, label=name, color=color)
axes[1].set_title("Распределение log10(интервал, мин)")
axes[1].set_xlabel("log10 минут"); axes[1].legend()
save_fig("11_inter_arrival_gaps", fig)
plt.show()

del gap    # временный фрейм на 9.5 млн строк
gc.collect()


In [ ]:
# ---- 4.5 Burst: сколько транзакций у счёта за один ЧАС ---------------
bh = df.groupby([COL_SENDER, "date", "hour"], observed=True)[COL_TARGET].agg(
    n="size", n_laundering="sum")
bh.columns = ["n_txn", "n_laundering"]
bh["bucket"] = bucketize(
    bh["n_txn"],
    bins=[1, 2, 3, 5, 10, 20, np.inf],
    labels=["1", "2", "3-4", "5-9", "10-19", "20+"],
)
burst = bh.groupby("bucket", observed=True).agg(
    n_windows=("n_txn", "size"), n_txn=("n_txn", "sum"), n_laundering=("n_laundering", "sum")
)
burst["rate_pct"] = burst["n_laundering"] / burst["n_txn"] * 100
burst["lift"] = (burst["n_laundering"] / burst["n_txn"]) / BASE_RATE
display(burst.round(3))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(burst.index.astype(str), burst["lift"], color="#64B5CD")
ax.axhline(1.0, color="grey", lw=1)
ax.set_title("Lift отмывания vs число транзакций счёта за час (burst)")
ax.set_xlabel("транзакций за час"); ax.set_ylabel("lift")
save_fig("12_hourly_burst", fig)
plt.show()

del bh
gc.collect()


In [ ]:
# ---- 4.6 Предпросмотр правила: «>= N платежей по 9-10k за один день» --
# Это мостик к Этапу 3 (rule-based baseline). Проверяем, что ТАКОЕ правило
# вообще имеет смысл на наших данных, до того как писать его в «продакшн-коде».

near_mask = df[COL_AMOUNT].between(band_lo, band_hi, inclusive="left")
near_by_day = near_mask.groupby([df[COL_SENDER], df["date"]], observed=True).sum()

for k in (2, 3, 5):
    suspicious = set(near_by_day.index.get_level_values(0)[near_by_day >= k])
    mask = df[COL_SENDER].isin(suspicious)
    print(f"\n=== Счёт делал >= {k} платежей в полосе {band_lo:,.0f}-{band_hi:,.0f} за один день ===")
    print(f"    таких счетов: {len(suspicious):,}")
    if len(suspicious) > 0:
        _ = compare_band(df, mask,
                         label_in=f">= {k} платежей {band_lo:,.0f}-{band_hi:,.0f} за день",
                         label_out="остальные")

> **Что получили.** Если lift растёт с ростом «серийности» — это прямое
> обоснование velocity-правил. Типичный порог в реальных банках:
> «≥3 операции на сумму 90–100% от порога за 24 часа одним клиентом» = алерт.
> Теперь у нас есть цифры, чтобы порог **обосновать**, а не выдумать.

---
# 5. География, коридоры, валюты, типы платежей

## Бизнес-контекст

**Коридор** = страна банка-отправителя → страна банка-получателя.
В реальном мониторинге коридоры — один из главных риск-факторов:

* есть юрисдикции повышенного риска (офшоры, страны из «серого списка» FATF);
* перевод «туда-обратно» между парой стран — типичный layering;
* **несовпадение валют** (платим USD → получаем EUR) — легальная конвертация,
  но ещё и способ разорвать цепочку сумм и запутать аудит.

**Метод:** для каждого коридора считаем объём, число отмываний, rate и lift.
**Обязательно фильтруем по минимальной поддержке** (`min_support`): коридор из
5 транзакций с 1 отмыванием даст rate 20% и lift 200 — это шум, а не риск.

In [ ]:
# ---- 5.1 Топ стран по объёму ----------------------------------------
for col, title in [(COL_SENDER_LOC, "Банк-отправитель"), (COL_RECEIVER_LOC, "Банк-получатель")]:
    loc = rate_by_group(df, col, min_support=1_000, sort_by="n")
    print(f"\n=== {title}: топ-10 по объёму ===")
    display(loc.head(10).round(3))

In [ ]:
# ---- 5.2 Таблица коридоров ------------------------------------------
MIN_SUPPORT = 1_000    # порог поддержки: меньше 1000 транзакций в коридоре не смотрим

corr = rate_by_group(df, [COL_SENDER_LOC, COL_RECEIVER_LOC],
                     min_support=MIN_SUPPORT, sort_by="n_laundering")

print("Топ-15 коридоров по АБСОЛЮТНОМУ числу отмываний (где теряются деньги):")
display(corr.head(15).round(3))

print("Топ-15 коридоров по LIFT (самые «грязные» относительно объёма):")
display(corr.sort_values("lift", ascending=False).head(15).round(3))

In [ ]:
# ---- 5.3 Тепловая карта: rate по коридорам --------------------------
top_senders = rate_by_group(df, COL_SENDER_LOC, min_support=1_000, sort_by="n").head(12).index
top_receivers = rate_by_group(df, COL_RECEIVER_LOC, min_support=1_000, sort_by="n").head(12).index

sub = df[df[COL_SENDER_LOC].isin(top_senders) & df[COL_RECEIVER_LOC].isin(top_receivers)]
heat = sub.groupby([COL_SENDER_LOC, COL_RECEIVER_LOC], observed=True)[COL_TARGET].agg(
    n="size", n_l="sum")
heat["rate_pct"] = heat["n_l"] / heat["n"] * 100
pivot = heat["rate_pct"].unstack(fill_value=np.nan)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Reds", linewidths=.4, ax=ax,
            cbar_kws={"label": "доля отмывания, %"})
ax.set_title("Доля отмывания (%) по коридорам (топ-12 стран с каждой стороны)")
ax.set_xlabel("Банк-получатель"); ax.set_ylabel("Банк-отправитель")
save_fig("13_corridor_heatmap", fig)
plt.show()

In [ ]:
# ---- 5.4 Кросс-бордер и несовпадение валют --------------------------
checks = {
    "кросс-бордер (страна отправителя != страна получателя)": df["is_cross_border"] == 1,
    "несовпадение валют (payment != received)": df["is_currency_mismatch"] == 1,
}

rows = []
for name, m in checks.items():
    n_in, x_in = int(m.sum()), int(df.loc[m, COL_TARGET].sum())
    n_out, x_out = int((~m).sum()), int(df.loc[~m, COL_TARGET].sum())
    z, p = two_proportion_ztest(x_in, n_in, x_out, n_out)
    rate_in, rate_out = x_in / n_in * 100, x_out / n_out * 100
    rows.append({"флаг": name, "n": n_in, "доля_%": n_in / len(df) * 100,
                 "rate_внутри_%": rate_in, "rate_вне_%": rate_out,
                 "lift": rate_in / rate_out, "p_value": p})
display(pd.DataFrame(rows).set_index("флаг").round(4))

# Топ валютных пар по lift
pairs_tbl = rate_by_group(df, [COL_PAY_CUR, COL_REC_CUR], min_support=1_000, sort_by="lift")
print("\nТоп-10 валютных пар по lift:")
display(pairs_tbl.head(10).round(3))

---
# 6. Типы платежей (Payment_type)

**Бизнес-логика:** разные каналы — разный риск.

* **Cash Deposit / Cash Withdrawal** — наличные: анонимность, точка входа placement'а.
  В реальных банках это самый «горячий» канал.
* **Wire / Cross-border** — быстрые крупные переводы, любимая дорожка для layering.
* **Cheque** — медленно, но легко запутать цепочку.
* **Credit/Debit card / ACH** — массовые каналы, шума много.

Проверяем, совпадает ли риск-логика с тем, что реально заложено в датасет.

In [ ]:
# ---- 6.1 Rate и lift по типам платежей -------------------------------
pay = rate_by_group(df, COL_PAY_TYPE, min_support=1_000, sort_by="lift")
display(pay.round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(pay.index[::-1], pay["n"][::-1], color=COLOR_OK)
axes[0].set_title("Объём транзакций по типу платежа")
axes[0].set_xlabel("n транзакций")

axes[1].barh(pay.index[::-1], pay["lift"][::-1], color=COLOR_BAD)
axes[1].axvline(1.0, color="grey", lw=1)
axes[1].set_title("Lift отмывания по типу платежа")
axes[1].set_xlabel("lift к base rate")
save_fig("14_payment_type_risk", fig)
plt.show()

---
# 7. Счета и граф: fan-in / fan-out (лёгкая версия, глубоко — на Этапе 2)

## Бизнес-контекст: зачем граф

Одна транзакция почти всегда выглядит «нормальной». **Схема** видна только на графе счетов:

```
   Fan-Out (раскидывание)        Fan-In (сбор)              Cycle (круг)
            A                      → A                       A → B
         /  |  \                 /   ↑                       ↑   ↓
        B   C   D               C    D                       D ← C
   один отправитель -> много   много -> один        A → B → C → D → A (вернулись)
```

* **Fan-Out** — дробление крупной суммы (structuring), типично для placement'а.
* **Fan-In** — сбор денег на один счёт (`Fan_In`, `Layered_Fan_In`, `Gather-Scatter`).
* **Cycle** — деньги возвращаются отправителю по кругу (`Cycle`).
* **Pass-through (транзитный счёт)** — и получает, и отправляет, почти ничего
  не оставляя себе. Классический «мул»/прокладка.
* **Bipartite / Stacked Bipartite** — две группы счетов, связанные «все со всеми».

Сейчас посмотрим на граф «глазами» (степени вершин), а полноценные graph-features
(циклы, компоненты связности, PageRank) сделаем на Этапе 2 в `src/graph_features.py`.

In [ ]:
# ---- 7.1 Степени вершин: сколько у счёта входящих и исходящих связей --
out_deg = df.groupby(COL_SENDER, observed=True).size().rename("out_degree")
in_deg = df.groupby(COL_RECEIVER, observed=True).size().rename("in_degree")

print(f"Уникальных счетов-отправителей: {len(out_deg):,}")
print(f"Уникальных счетов-получателей:  {len(in_deg):,}")
print("\nИсходящая степень (сколько платежей отправляет счёт):")
display(out_deg.describe(percentiles=[.5, .9, .99, .999]).to_frame().T)
print("Входящая степень (сколько платежей получает счёт):")
display(in_deg.describe(percentiles=[.5, .9, .99, .999]).to_frame().T)

# Степени обычно распределены по степенному закону: много «спящих» счетов
# и немного сверх-активных хабов. Проверим на графике.
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].hist(np.log10(out_deg.clip(lower=1)), bins=60, color=COLOR_OK)
axes[0].set_title("log10(out-degree) — сколько платежей отправляет счёт")
axes[0].set_xlabel("log10(число исходящих транзакций)")
axes[1].hist(np.log10(in_deg.clip(lower=1)), bins=60, color=COLOR_BAD)
axes[1].set_title("log10(in-degree) — сколько платежей получает счёт")
axes[1].set_xlabel("log10(число входящих транзакций)")
save_fig("15_degree_distributions", fig)
plt.show()

In [ ]:
# ---- 7.2 Зависимость риска от «ветвистости» счёта --------------------
# transform("size") — посчитать размер группы и развернуть его на каждую строку.
df["sender_out_degree"] = df.groupby(COL_SENDER, observed=True)[COL_TARGET].transform("size")
df["receiver_in_degree"] = df.groupby(COL_RECEIVER, observed=True)[COL_TARGET].transform("size")

deg_bins = [1, 2, 5, 10, 25, 50, 100, 500, np.inf]
deg_labels = ["1", "2-4", "5-9", "10-24", "25-49", "50-99", "100-499", "500+"]

# Fan-Out: риск транзакции vs сколько всего транзакций отправил этот счёт
fanout = rate_by_bucket(
    df.assign(_b=bucketize(df["sender_out_degree"], deg_bins, deg_labels)), "_b")
# Fan-In: риск vs сколько транзакций пришло на счёт получателя
fanin = rate_by_bucket(
    df.assign(_b=bucketize(df["receiver_in_degree"], deg_bins, deg_labels)), "_b")

display(pd.concat({"fan-out (по out-degree отправителя)": fanout,
                   "fan-in (по in-degree получателя)": fanin}, axis=1).round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
axes[0].bar(fanout.index.astype(str), fanout["lift"], color=COLOR_OK)
axes[0].axhline(1.0, color="grey", lw=1)
axes[0].set_title("Fan-Out: lift vs число транзакций ОТ счёта")
axes[0].set_xlabel("out-degree"); axes[0].set_ylabel("lift")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(fanin.index.astype(str), fanin["lift"], color=COLOR_BAD)
axes[1].axhline(1.0, color="grey", lw=1)
axes[1].set_title("Fan-In: lift vs число транзакций НА счёт")
axes[1].set_xlabel("in-degree"); axes[1].set_ylabel("lift")
axes[1].tick_params(axis="x", rotation=30)
save_fig("16_fanin_fanout_lift", fig)
plt.show()

In [ ]:
# ---- 7.3 Транзитные счета (pass-through) -----------------------------
# Счёт и получает, и отправляет = потенциальная «прокладка» в цепочке layering.
senders = set(df[COL_SENDER].unique())
receivers = set(df[COL_RECEIVER].unique())
both = senders & receivers

print(f"Счетов-отправителей: {len(senders):,}")
print(f"Счетов-получателей:  {len(receivers):,}")
print(f"И тех, и других (транзитных): {len(both):,} "
      f"({len(both) / max(len(senders), 1) * 100:.1f}% от отправителей)")

_ = compare_band(df, df[COL_SENDER].isin(both),
                 label_in="отправитель — транзитный счёт",
                 label_out="отправитель только отправляет")

In [ ]:
# ---- 7.4 Граф-паттерны: повторяющиеся пары и mutual-пары (цикл длины 2) --
# Сколько раз встречалась пара (A -> B): частые повторы = «накатанная дорожка».
df["pair_frequency"] = df.groupby([COL_SENDER, COL_RECEIVER], observed=True)[COL_TARGET].transform("size")
pair_tbl = rate_by_bucket(
    df.assign(_b=bucketize(df["pair_frequency"],
                           [1, 2, 3, 5, 10, 25, np.inf],
                           ["1", "2", "3-4", "5-9", "10-24", "25+"])), "_b")
display(pair_tbl.round(3))

# Mutual-пары: есть и A->B, и B->A (простейший цикл: деньги вернулись)
pairs = df[[COL_SENDER, COL_RECEIVER]].drop_duplicates()
reversed_pairs = pairs.rename(columns={COL_SENDER: COL_RECEIVER, COL_RECEIVER: COL_SENDER})
mutual = pairs.merge(reversed_pairs, on=[COL_SENDER, COL_RECEIVER])
mutual = mutual.assign(is_mutual=1)[[COL_SENDER, COL_RECEIVER, "is_mutual"]]

print(f"Уникальных направленных пар: {len(pairs):,}")
print(f"Взаимных пар (A->B и B->A):  {len(mutual):,}")

df_merged = df[[COL_SENDER, COL_RECEIVER, COL_TARGET]].merge(
    mutual, on=[COL_SENDER, COL_RECEIVER], how="left")
_ = compare_band(df_merged, df_merged["is_mutual"].fillna(0) == 1,
                 label_in="транзакция во взаимной паре (цикл длины 2)",
                 label_out="односторонняя пара")

del pairs, reversed_pairs, mutual, df_merged
gc.collect()


In [ ]:
# ---- 7.5 Концентрация: не крутится ли всё вокруг пары процентов счетов --
for col, name in [(COL_SENDER, "отправители"), (COL_RECEIVER, "получатели")]:
    totals = df.groupby(col, observed=True)[COL_AMOUNT].sum().sort_values(ascending=False)
    for pct in (0.01, 0.05):
        k = max(1, int(len(totals) * pct))
        share = totals.head(k).sum() / totals.sum() * 100
        print(f"Топ-{pct:.0%} {name} ({k:,} счетов) держат {share:.1f}% всего оборота")

illicit_by_sender = laund.groupby(COL_SENDER, observed=True).size().sort_values(ascending=False)
print(f"\nСчетов, участвовавших в отмывании: {len(illicit_by_sender):,}")
print("Топ-10 счетов по числу отмывочных транзакций (счёт: число операций):")
print(illicit_by_sender.head(10).to_string())
top1 = illicit_by_sender.head(max(1, len(illicit_by_sender) // 100))
print(f"На топ-1% таких счетов приходится {top1.sum() / illicit_by_sender.sum() * 100:.1f}% "
      f"всех отмывочных операций")

---
# 8. Качество данных

Мелочь, которая отличает взрослый проект: перед тем как строить признаки,
проверяем данные на артефакты. В реале здесь находят и дубли, и нулевые суммы,
и самопереводы — всё это потом «взрывает» модель или даёт липовые алерты.

In [ ]:
# ---- 8.1 Проверки качества ------------------------------------------
# Полный df.duplicated() по ВСЕМ колонкам на 9.5 млн строк требует много памяти,
# поэтому включаем его только если данных «немного». На большом объёме достаточно
# проверки по бизнес-ключу (дата + время + отправитель + получатель + сумма).
FULL_DUPLICATE_CHECK = len(df) <= 3_000_000

checks = {
    "Полные дубли строк (все колонки)": (int(df.duplicated().sum())
                                         if FULL_DUPLICATE_CHECK else "пропущено: >3 млн строк"),
    "Дубли по (дата, время, отправитель, получатель, сумма)": int(
        df.duplicated(subset=[COL_DATE, "hour", "minute", COL_SENDER, COL_RECEIVER,
                                  COL_AMOUNT]).sum()),
    "Самопереводы (отправитель == получатель)": int(
        categories_equal(df[COL_SENDER], df[COL_RECEIVER]).sum()),
    "Сумма <= 0": int((df[COL_AMOUNT] <= 0).sum()),
    "Сумма == 0": int((df[COL_AMOUNT] == 0).sum()),
    "Пропуски в Amount": int(df[COL_AMOUNT].isna().sum()),
    "Пропуски в hour": int(df["hour"].isna().sum()) if "hour" in df else 0,
    "Пропуски в date": int(df["date"].isna().sum()) if "date" in df else 0,
}
display(pd.DataFrame({"проверка": list(checks.keys()), "n": list(checks.values())})
        .set_index("проверка"))

# ---- 8.2 Корреляции числовых колонок --------------------------------
num_cols = [c for c in [COL_AMOUNT, "amount_pct_in_currency", "hour", "amount_vs_own_median",
                        "sender_out_degree", "receiver_in_degree", "pair_frequency",
                        "is_cross_border", "is_currency_mismatch", COL_TARGET]
            if c in df.columns]
corr_m = df[num_cols].sample(min(len(df), 500_000), random_state=RANDOM_STATE).corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_m, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax,
            annot_kws={"size": 8})
ax.set_title("Корреляции числовых признаков (выборка, включая target)")
save_fig("17_correlations", fig)
plt.show()

print("\nКорреляция с Is_laundering (по модулю):")
display(corr_m[COL_TARGET].drop(COL_TARGET).abs().sort_values(ascending=False).to_frame("|corr|"))

> **Важное предупреждение про этот график корреляций.**
> Скорее всего все корреляции окажутся около нуля (0.00–0.05). Это НЕ значит,
> что признаки бесполезны! При сильнейшем дисбалансе (0.1%) линейная корреляция
> Пирсона почти всегда около нуля — её «гасит» разброс. Отмывание ловится
> **нелинейными** моделями (XGBoost/LightGBM) и **порогами/комбинациями**
> (например: «сумма 9-10k **И** 3+ транзакции за час **И** кросс-бордер»).
> Именно поэтому на Этапе 4 будет градиентный бустинг, а не логистическая регрессия.

---
# 9. Сохранение результатов и черновик выводов

Финальные ячейки:
1. сохраняют подготовленные данные (чтобы Этап 2 не перечитывал CSV);
2. собирают **числа** и печатают готовый markdown-блок для README.

Переноси выводы в `README.md` и в презентацию — это и есть «мясо» для собеседования:
не «я построил модель», а **«я нашёл в данных structuring у порога $10k с lift 3.2,
p < 0.001, и на этом построил правило»**.

In [ ]:
# ---- 9.1 Сохраняем подготовленный датафрейм -------------------------
# Зачем: Этап 2 (feature engineering) начнётся с load_dataset() и не будет
# заново парсить 9.5 млн строк CSV.
out_path = save_table(df, PROCESSED_DIR / "saml_d_eda_enriched.pkl")
print(f"\nСохранено: {out_path}")
print(f"Колонок теперь: {df.shape[1]}")
print(list(df.columns))

In [ ]:
# ---- 9.2 Автосборка сводки для README -------------------------------
# Пересчитываем ключевые цифры заново, чтобы сводка была корректной
# независимо от того, какие ячейки выше запускались.
#
# Ключевой принцип: для КАЖДОГО вывода считаем и lift, и p-value.
# Красивый lift без проверки значимости — это не анализ, а гадание.

total_l = int(df[COL_TARGET].sum())


def lift_and_p(x_in: int, n_in: int) -> tuple[float, float]:
    """lift группы «внутри» к base rate + p-value против «всех остальных»."""
    z, p = two_proportion_ztest(x_in, n_in, total_l - x_in, len(df) - n_in)
    rate_in = x_in / n_in if n_in else np.nan
    return (rate_in / BASE_RATE if BASE_RATE else np.nan), p


# 1) Structuring: полоса 90-100% от порога $10 000
near_m = df[COL_AMOUNT].between(band_lo, band_hi, inclusive="left")
lift_near, p_near = lift_and_p(int(df.loc[near_m, COL_TARGET].sum()), int(near_m.sum()))

# 2) Кросс-бордер
cross = df["is_cross_border"] == 1
lift_cross, p_cross = lift_and_p(int(df.loc[cross, COL_TARGET].sum()), int(cross.sum()))

# 3) Топ-коридор по lift (с порогом поддержки, иначе всплывёт шум)
top_corridor = rate_by_group(df, [COL_SENDER_LOC, COL_RECEIVER_LOC],
                             min_support=1_000, sort_by="lift").head(1)
lift_cor, p_cor = lift_and_p(int(top_corridor["n_laundering"].iloc[0]),
                             int(top_corridor["n"].iloc[0]))

# 4) Топ-канал по lift
top_pay = rate_by_group(df, COL_PAY_TYPE, min_support=1_000, sort_by="lift").head(1)
lift_pay, p_pay = lift_and_p(int(top_pay["n_laundering"].iloc[0]), int(top_pay["n"].iloc[0]))

# 5) Velocity: минимальная дневная активность, где lift впервые >= 3.
#    Максимальный lift брать нельзя: на крошечной группе он упирается в 1/base_rate
#    (здесь это 962x) и не пригоден как порог для правила.
vel_thr = vel[vel["lift"] >= 3].index.min() if (vel["lift"] >= 3).any() else "не достигнут"
if vel_thr in vel.index:
    lift_vel, p_vel = lift_and_p(int(vel.loc[vel_thr, "n_laundering"]),
                                 int(vel.loc[vel_thr, "n_txn"]))
else:
    lift_vel, p_vel = np.nan, 1.0

summary = {
    "n_transactions":      len(df),
    "n_accounts":          len(set(df[COL_SENDER].unique()) | set(df[COL_RECEIVER].unique())),
    "period":              (f"{df['date'].min().date()} – {df['date'].max().date()}"
                            if "date" in df else "n/a"),
    "base_rate_pct":       BASE_RATE * 100,
    "n_laundering":        total_l,
    "n_typologies":        int(laund[COL_LAUND_TYPE].nunique()),
    "top_typologies":      ", ".join(typ.head(3).index.tolist()),
    "structuring_band":    f"{band_lo:,.0f}–{band_hi:,.0f}",
    "structuring_lift":    lift_near,
    "structuring_p":       p_near,
    "cross_border_lift":   lift_cross,
    "cross_border_p":      p_cross,
    "cross_border_share":  cross.mean() * 100,
    "top_corridor":        " -> ".join(map(str, top_corridor.index[0])),
    "top_corridor_lift":   lift_cor,
    "top_corridor_p":      p_cor,
    "top_payment_type":    str(top_pay.index[0]),
    "top_payment_lift":    lift_pay,
    "top_payment_p":       p_pay,
    "velocity_threshold":  str(vel_thr),
    "velocity_lift":       lift_vel,
    "velocity_p":          p_vel,
}

for k, v in summary.items():
    print(f"{k:22s}: {v}")

In [ ]:
# ---- 9.3 Готовый markdown-блок для README (выводы следуют из цифр) ----
# Главное правило аналитика: вывод должен СЛЕДОВАТЬ из цифр, а не подгоняться
# под ожидания. Поэтому формулировки зависят от lift и p-value:
#     lift >= 1.5 и p < 0.05 -> сигнал подтверждён (годится в правило);
#     1.2 <= lift < 1.5      -> слабый сигнал (только как признак для модели);
#     иначе                  -> связи нет: признак не использовать.

def verdict(name: str, lift: float, p: float, share: float | None = None,
            note: str = "") -> str:
    share_txt = f" (доля таких операций {share:.1f}%)" if share is not None else ""
    if p >= 0.05:
        return (f"**{name}: связи не обнаружено** — lift = {lift:.2f}x{share_txt}, "
                f"p-value = {p:.2f}. Различие статистически незначимо: как самостоятельный "
                f"красный флаг признак не годится.{note}")
    if lift >= 1.5:
        return (f"**{name}: сигнал подтверждён** — lift = **{lift:.2f}x**{share_txt}, "
                f"p-value = {p:.1e}.{note}")
    if lift >= 1.2:
        return (f"**{name}: слабый сигнал** — lift = {lift:.2f}x{share_txt}, "
                f"p-value = {p:.1e}. Годится как признак для модели, но не как жёсткое "
                f"правило.{note}")
    return (f"**{name}: сигнал не подтверждён** — lift = {lift:.2f}x{share_txt}, "
            f"p-value = {p:.1e}, то есть риск внутри группы такой же, как в среднем "
            f"по выборке.{note}")


conclusions = [
    verdict(f"Structuring: сумма в полосе {summary['structuring_band']}",
            summary["structuring_lift"], summary["structuring_p"],
            note="  Классика placement: дробление ниже порога обязательной отчётности (CTR)."),
    verdict("Кросс-бордер (страна отправителя != страна получателя)",
            summary["cross_border_lift"], summary["cross_border_p"],
            summary["cross_border_share"],
            note="  Если lift ~ 1 — в этом датасете география не разделяет риск."),
    verdict(f"Коридор {summary['top_corridor']}",
            summary["top_corridor_lift"], summary["top_corridor_p"],
            note="  Лучший коридор по lift при поддержке >= 1 000 транзакций."),
    verdict(f"Канал {summary['top_payment_type']}",
            summary["top_payment_lift"], summary["top_payment_p"],
            note="  Тип платежа — самый дешёвый признак: он есть в любой АБС-системе."),
    verdict(f"Velocity: {summary['velocity_threshold']} транзакций в день на счёт",
            summary["velocity_lift"], summary["velocity_p"],
            note="  Серийность — ядро smurfing/structuring."),
]

readme_block = f"""
## Ключевые находки EDA (SAML-D)

| Метрика | Значение |
|---|---|
| Транзакций | {summary['n_transactions']:,} |
| Уникальных счетов | {summary['n_accounts']:,} |
| Период | {summary['period']} |
| Отмывание (Is_laundering=1) | {summary['n_laundering']:,} ({summary['base_rate_pct']:.4f}%) |
| Типологий отмывания | {summary['n_typologies']} |
| Топ-3 типологии | {summary['top_typologies']} |

**Выводы (сформулированы автоматически по результатам z-теста двух долей):**

""" + "\n".join(f"{k}. {c}" for k, c in enumerate(conclusions, 1)) + f"""
{len(conclusions) + 1}. **Данные имеют временную структуру** -> для ML используется только
   time-based train/test split: иначе модель «увидит будущее» и метрики окажутся завышены.

> Как читать lift: 1.0x = такой же риск, как в среднем по выборке;
> 3x = группа втрое «грязнее» случайной транзакции.
> p-value отвечает на вопрос «а не случайность ли это?».
"""

print(readme_block)

reports_dir = Path(PROJECT_ROOT) / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
(reports_dir / "eda_findings.md").write_text(readme_block, encoding="utf-8")
print("Сохранено в reports/eda_findings.md")

In [ ]:
# ---- 9.4 Чек-лист Этапа 1 -------------------------------------------
checklist = [
    "[x] Данные загружены, типы оптимизированы, пропуски проверены",
    f"[x] Дисбаланс классов измерен: base rate = {BASE_RATE * 100:.4f}%",
    "[x] Раскладка по 17 типологиям отмывания построена",
    "[x] Проверена гипотеза structuring у порога $10 000 (lift + z-тест)",
    "[x] Проверены «круглые» суммы и отклонение от обычного поведения счёта",
    "[x] Проанализированы час / день недели / календарный тренд",
    "[x] Построены velocity-метрики: день, час, inter-arrival",
    "[x] Построены коридоры (lift по странам, валютам, каналам)",
    "[x] Оценены граф-паттерны: fan-in/fan-out, транзитные счета, mutual-пары",
    "[x] Проверено качество данных и корреляции",
    "[x] Итоги сохранены в reports/eda_findings.md и reports/figures/",
]
print("\n".join(checklist))
print("\nЭТАП 1 ГОТОВ. Дальше: Этап 2 — Feature Engineering "
      "(src/features.py + src/graph_features.py).")